# Capstone 1 — Data Import and Cleaning

**Aura / ClickO healthcare prep**  
Dataset: `NSMES1988.csv`

Tasks: import, inspect, missing values, age/income notes, JSON export, memory / dtypes, export `NSMES1988new.csv`, short report.


## Step 1 — Import libraries

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA = Path('NSMES1988.csv')
print('pandas', pd.__version__, '| numpy', np.__version__)


pandas 3.0.6 | numpy 2.5.3


## Step 2 — Load CSV into a dataframe

In [2]:
# first column in the file is an unnamed index — drop it after load
raw = pd.read_csv(DATA)
print('columns:', list(raw.columns))
df = raw.copy()
if df.columns[0].startswith('Unnamed'):
    df = df.drop(columns=df.columns[0])
df.head()


columns: ['Unnamed: 0', 'visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'health', 'chronic', 'adl', 'region', 'age', 'gender', 'married', 'school', 'income', 'employed', 'insurance', 'medicaid']


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
0,5,0,0,0,0,1,average,2,normal,other,6.9,male,yes,6,2.8810,yes,yes,no
1,1,0,2,0,2,0,average,2,normal,other,7.4,female,yes,10,2.7478,no,yes,no
2,13,0,0,0,3,3,poor,4,limited,other,6.6,female,no,10,0.6532,no,no,yes
3,16,0,5,0,1,1,poor,2,limited,other,7.6,male,yes,3,0.6588,no,yes,no
4,3,0,0,0,0,0,average,2,limited,other,7.9,female,yes,6,0.6588,no,yes,no


## Step 3 — Inspect rows, columns, dtypes

In [3]:
print('shape (rows, cols):', df.shape)
print()
print('dtypes:')
print(df.dtypes)
print()
df.info()
df.tail(3)


shape (rows, cols): (4406, 18)

dtypes:
visits         int64
nvisits        int64
ovisits        int64
novisits       int64
emergency      int64
hospital       int64
health           str
chronic        int64
adl              str
region           str
age          float64
gender           str
married          str
school         int64
income       float64
employed         str
insurance        str
medicaid         str
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 4406 entries, 0 to 4405
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   visits     4406 non-null   int64  
 1   nvisits    4406 non-null   int64  
 2   ovisits    4406 non-null   int64  
 3   novisits   4406 non-null   int64  
 4   emergency  4406 non-null   int64  
 5   hospital   4406 non-null   int64  
 6   health     4406 non-null   str    
 7   chronic    4406 non-null   int64  
 8   adl        4406 non-null   str    
 9   region     4406 non-null  

,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
4403,10,0,20,0,1,1,average,5,normal,other,7.3,male,yes,12,3.877916,no,yes,no
4404,16,1,0,0,0,0,average,0,normal,other,6.6,female,yes,12,3.877916,no,yes,no
4405,0,0,0,0,0,0,excellent,0,normal,other,7.1,male,yes,0,6.596800,yes,no,no


## Step 4 — Missing values (is the data clean?)

In [4]:
missing = df.isna().sum()
print(missing)
print()
print('total missing cells:', int(missing.sum()))
print('duplicate rows:', int(df.duplicated().sum()))
# note: zero missing means the frame looks complete for nulls


visits       0
nvisits      0
ovisits      0
novisits     0
emergency    0
hospital     0
health       0
chronic      0
adl          0
region       0
age          0
gender       0
married      0
school       0
income       0
employed     0
insurance    0
medicaid     0
dtype: int64

total missing cells: 0
duplicate rows: 0


## Step 5 — Comments on age and income (values + range)

In [5]:
age = df['age']
income = df['income']
print('AGE — stored as years/10 per data dictionary')
print('  min/max/mean:', age.min(), age.max(), round(age.mean(), 3))
print('  approx real years: min/max', age.min()*10, age.max()*10)
print()
print('INCOME — family income in USD 10000 units')
print('  min/max/mean:', income.min(), income.max(), round(income.mean(), 3))
print('  approx USD: min/max', income.min()*10000, income.max()*10000)
print()
df[['age', 'income']].describe()


AGE — stored as years/10 per data dictionary
  min/max/mean: 6.6 10.9 7.402
  approx real years: min/max 66.0 109.0

INCOME — family income in USD 10000 units
  min/max/mean: -1.0125 54.8351 2.527
  approx USD: min/max -10125.0 548351.0



,age,income
count,4406.000000,4406.000000
mean,7.402406,2.527132
std,0.633405,2.924648
min,6.600000,-1.012500
25%,6.900000,0.912150
50%,7.300000,1.698150
75%,7.800000,3.172850
max,10.900000,54.835100


### Notes (age / income)
- `age` is scaled (years ÷ 10). Example: 6.9 ≈ 69 years.
- `income` is in units of USD 10,000. Example: 2.881 ≈ $28,810.
- Keep these scales in mind until Capstone 2 (where we multiply back).


## Step 6 — Export JSON and review

In [6]:
json_path = Path('NSMES1988.json')
# orient=records is easy to skim; keep it readable
df.to_json(json_path, orient='records', indent=2)
print('wrote', json_path, 'size_bytes=', json_path.stat().st_size)
# peek first record
with open(json_path, encoding='utf-8') as f:
    sample = json.load(f)[:1]
sample


wrote NSMES1988.json size_bytes= 1574860


[{'visits': 5,
  'nvisits': 0,
  'ovisits': 0,
  'novisits': 0,
  'emergency': 0,
  'hospital': 1,
  'health': 'average',
  'chronic': 2,
  'adl': 'normal',
  'region': 'other',
  'age': 6.9,
  'gender': 'male',
  'married': 'yes',
  'school': 6,
  'income': 2.881,
  'employed': 'yes',
  'insurance': 'yes',
  'medicaid': 'no'}]

### JSON comments
- Export works; structure is a list of row objects.
- File is larger than CSV (field names repeated). Fine for exchange, CSV is better for reuse here.


## Step 7 — Memory usage + dtype recommendations

In [7]:
mem = df.memory_usage(deep=True)
print(mem)
print()
print('total deep memory (bytes):', int(mem.sum()))
print('total deep memory (MB):', round(mem.sum() / 1e6, 3))

# recommendations (do not apply yet — Capstone asks recommend first)
cats = ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']
ints = ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'chronic', 'school']
print()
print('Recommend category for:', cats)
print('Recommend smaller ints (int16/int32) for:', ints)
print('age/income can stay float32 for memory if precision is enough')


Index           132
visits        35248
nvisits       35248
ovisits       35248
novisits      35248
emergency     35248
hospital      35248
health       245760
chronic       35248
adl          243229
region       242788
age           35248
gender       238774
married      227112
school        35248
income        35248
employed     225161
insurance    228127
medicaid     225108
dtype: int64

total deep memory (bytes): 2228671
total deep memory (MB): 2.229

Recommend category for: ['health', 'adl', 'region', 'gender', 'married', 'employed', 'insurance', 'medicaid']
Recommend smaller ints (int16/int32) for: ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'chronic', 'school']
age/income can stay float32 for memory if precision is enough


### Recommended dataframe changes before detailed analysis
1. Drop leftover index column if present (already done above).
2. Convert factor columns to `category` (health, region, gender, married, employed, insurance, medicaid, adl).
3. Downcast count columns to smaller integer types.
4. Document that age/income are scaled — either keep a data dictionary note or create helper columns later.
5. Capstone 2 will rescale age × 10 and income × 10000.


## Step 8 — Export cleaned CSV for later sessions

In [8]:
out_csv = Path('NSMES1988new.csv')
df.to_csv(out_csv, index=False)
print('wrote', out_csv, 'rows=', len(df), 'cols=', df.shape[1])
pd.read_csv(out_csv).head(2)


wrote NSMES1988new.csv rows= 4406 cols= 18


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid
0,5,0,0,0,0,1,average,2,normal,other,6.9,male,yes,6,2.8810,yes,yes,no
1,1,0,2,0,2,0,average,2,normal,other,7.4,female,yes,10,2.7478,no,yes,no


## Step 9 — Short visual / structural report

**Observations**
- Healthcare survey-style frame: visit counts + demographics + insurance flags.
- No missing values in this extract; good starting point for Aura aggregation practice.
- Mix of numeric counts and string factors — factors should become categories before heavy analysis.
- Age and income are intentionally scaled in the source file; don’t treat raw values as calendar years or raw dollars until rescaled.
- Ready for Capstone 2 processing using `NSMES1988new.csv`.
